# SC-13-Solana-Anchor - Solana avec Anchor

**Navigation** : [Index](../README.md) | [<< Move Sui](SC-12-Move-Sui.ipynb) | [Cross-Chain >>](SC-14-Cross-Chain.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre l'architecture **Solana** (accounts, programs)
2. Utiliser le framework **Anchor** (Rust)
3. Creer des **PDAs** (Program Derived Addresses)
4. Implementer des **CPIs** (Cross-Program Invocations)

### Duree estimee : 55 minutes

---

## 1. Architecture Solana

Solana utilise un modele different d'Ethereum.

In [1]:
# Comparaison Ethereum vs Solana
print("""
ETHEREUM vs SOLANA

| Concept       | Ethereum              | Solana                  |
|--------------|----------------------|-------------------------|
| Execution    | EVM                   | BPF (Berkeley Packet Filter) |
| State        | Storage in contract  | Accounts (separates)    |
| Langage      | Solidity              | Rust (via Anchor)       |
| Adressage    | Nonces sequentielles  | Hash d'accounts         |
| Fees         | Gas (variable)        | Compute units + rent    |
| TPS          | ~15-30                | ~65,000                 |

COMPOSANTS SOLANA:
- Program    : Code executable (stateless)
- Account    : Stockage de donnees (state)
- PDA        : Program Derived Address
- CPI        : Cross-Program Invocation
- Token      : SPL Token (equivalent ERC-20/721)
""")


ETHEREUM vs SOLANA

| Concept       | Ethereum              | Solana                  |
|--------------|----------------------|-------------------------|
| Execution    | EVM                   | BPF (Berkeley Packet Filter) |
| State        | Storage in contract  | Accounts (separates)    |
| Langage      | Solidity              | Rust (via Anchor)       |
| Adressage    | Nonces sequentielles  | Hash d'accounts         |
| Fees         | Gas (variable)        | Compute units + rent    |
| TPS          | ~15-30                | ~65,000                 |

COMPOSANTS SOLANA:
- Program    : Code executable (stateless)
- Account    : Stockage de donnees (state)
- PDA        : Program Derived Address
- CPI        : Cross-Program Invocation
- Token      : SPL Token (equivalent ERC-20/721)



---

## 2. Programme Anchor Simple

In [2]:
# Counter en Anchor
ANCHOR_COUNTER = '''
// Fichier: programs/counter/src/lib.rs

use anchor_lang::prelude::*;

declare_id!("Counter11111111111111111111111111111111111");

#[program]
pub mod counter {
    use super::*;

    pub fn initialize(ctx: Context<Initialize>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        counter.count = 0;
        counter.authority = ctx.accounts.authority.key();
        Ok(())
    }

    pub fn increment(ctx: Context<Increment>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        counter.count += 1;
        Ok(())
    }

    pub fn decrement(ctx: Context<Decrement>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        require!(counter.count > 0, CounterError::Underflow);
        counter.count -= 1;
        Ok(())
    }
}

#[derive(Accounts)]
pub struct Initialize<'info> {
    #[account(
        init,
        payer = authority,
        space = 8 + Counter::INIT_SPACE,
        seeds = [b"counter", authority.key().as_ref()],
        bump
    )]
    pub counter: Account<'info, Counter>,
    #[account(mut)]
    pub authority: Signer<'info>,
    pub system_program: Program<'info, System>,
}

#[derive(Accounts)]
pub struct Increment<'info> {
    #[account(
        mut,
        seeds = [b"counter", authority.key().as_ref()],
        bump
    )]
    pub counter: Account<'info, Counter>,
    pub authority: Signer<'info>,
}

#[derive(Accounts)]
pub struct Decrement<'info> {
    #[account(
        mut,
        seeds = [b"counter", authority.key().as_ref()],
        bump
    )]
    pub counter: Account<'info, Counter>,
    pub authority: Signer<'info>,
}

#[account]
#[derive(InitSpace)]
pub struct Counter {
    pub authority: Pubkey,
    pub count: u64,
}

#[error_code]
pub enum CounterError {
    #[msg("Counter cannot go below zero")]
    Underflow,
}
'''

print("Programme Counter Anchor:")
print(ANCHOR_COUNTER)

Programme Counter Anchor:

// Fichier: programs/counter/src/lib.rs

use anchor_lang::prelude::*;

declare_id!("Counter11111111111111111111111111111111111");

#[program]
pub mod counter {
    use super::*;

    pub fn initialize(ctx: Context<Initialize>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        counter.count = 0;
        counter.authority = ctx.accounts.authority.key();
        Ok(())
    }

    pub fn increment(ctx: Context<Increment>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        counter.count += 1;
        Ok(())
    }

    pub fn decrement(ctx: Context<Decrement>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        require!(counter.count > 0, CounterError::Underflow);
        counter.count -= 1;
        Ok(())
    }
}

#[derive(Accounts)]
pub struct Initialize<'info> {
    #[account(
        init,
        payer = authority,
        space = 8 + Counter::INIT_SPACE,
        seeds = [b"counter", authority.k

---

## 3. PDAs (Program Derived Addresses)

Les PDAs sont des adresses derivees deterministes.

In [3]:
# PDA explanation
print("""
PDA (Program Derived Address)

PROPRIETES:
- Adresse derivee du program_id et de seeds
- Pas de cle privee (programme seulement)
- Deterministe: meme seeds = meme PDA
- Bump: [b"seed", bump] pour trouver le PDA

GENERATION:
seeds = [b"counter", authority.key().as_ref()]
(pda, bump) = Pubkey::find_program_address(&seeds, program_id)

UTILISATION:
1. Stockage de donnees utilisateur
2. Escrows
3. Token accounts
4. Metadata

EXEMPLE DANS ANCHOR:
#[account(
    seeds = [b"counter", authority.key().as_ref()],
    bump
)]
pub counter: Account<'info, Counter>,
""")


PDA (Program Derived Address)

PROPRIETES:
- Adresse derivee du program_id et de seeds
- Pas de cle privee (programme seulement)
- Deterministe: meme seeds = meme PDA
- Bump: [b"seed", bump] pour trouver le PDA

GENERATION:
seeds = [b"counter", authority.key().as_ref()]
(pda, bump) = Pubkey::find_program_address(&seeds, program_id)

UTILISATION:
1. Stockage de donnees utilisateur
2. Escrows
3. Token accounts
4. Metadata

EXEMPLE DANS ANCHOR:
#[account(
    seeds = [b"counter", authority.key().as_ref()],
    bump
)]
pub counter: Account<'info, Counter>,



---

## 4. CPI (Cross-Program Invocation)

In [4]:
# CPI example
CPI_EXAMPLE = '''
// CPI vers System Program pour transfer SOL
use anchor_lang::system_program;

pub fn transfer_sol(ctx: Context<TransferSol>, amount: u64) -> Result<()> {
    let cpi_context = CpiContext::new(
        ctx.accounts.system_program.to_account_info(),
        system_program::Transfer {
            from: ctx.accounts.from.to_account_info(),
            to: ctx.accounts.to.to_account_info(),
        }
    );
    system_program::transfer(cpi_context, amount)
}

#[derive(Accounts)]
pub struct TransferSol<'info> {
    #[account(mut)]
    pub from: Signer<'info>,
    #[account(mut)]
    pub to: AccountInfo<'info>,
    pub system_program: Program<'info, System>,
}

// CPI vers Token Program
use anchor_spl::token::{self, Transfer as TokenTransfer};

pub fn transfer_token(
    ctx: Context<TransferToken>,
    amount: u64
) -> Result<()> {
    let cpi_accounts = TokenTransfer {
        from: ctx.accounts.from.to_account_info(),
        to: ctx.accounts.to.to_account_info(),
        authority: ctx.accounts.authority.to_account_info(),
    };
    let cpi_program = ctx.accounts.token_program.to_account_info();
    token::transfer(CpiContext::new(cpi_program, cpi_accounts), amount)
}
'''

print("CPI Examples:")
print(CPI_EXAMPLE)

CPI Examples:

// CPI vers System Program pour transfer SOL
use anchor_lang::system_program;

pub fn transfer_sol(ctx: Context<TransferSol>, amount: u64) -> Result<()> {
    let cpi_context = CpiContext::new(
        ctx.accounts.system_program.to_account_info(),
        system_program::Transfer {
            from: ctx.accounts.from.to_account_info(),
            to: ctx.accounts.to.to_account_info(),
        }
    );
    system_program::transfer(cpi_context, amount)
}

#[derive(Accounts)]
pub struct TransferSol<'info> {
    #[account(mut)]
    pub from: Signer<'info>,
    #[account(mut)]
    pub to: AccountInfo<'info>,
    pub system_program: Program<'info, System>,
}

// CPI vers Token Program
use anchor_spl::token::{self, Transfer as TokenTransfer};

pub fn transfer_token(
    ctx: Context<TransferToken>,
    amount: u64
) -> Result<()> {
    let cpi_accounts = TokenTransfer {
        from: ctx.accounts.from.to_account_info(),
        to: ctx.accounts.to.to_account_info(),
        a

---

## 5. Commandes Anchor CLI

In [5]:
# Anchor CLI commands
print("""
INSTALLATION:
cargo install --git https://github.com/coral-xyz/anchor avm --locked --force
avm install latest
avm use latest

CREATION PROJET:
anchor init my_program
cd my_program

STRUCTURE:
my_program/
|-- Anchor.toml          # Configuration
|-- Cargo.toml
|-- programs/
|   `-- my_program/
|       |-- Cargo.toml
|       `-- src/
|           `-- lib.rs   # Code principal
|-- tests/
|   `-- my_program.ts    # Tests
`-- app/                 # Frontend

COMMANDES:
anchor build              # Compiler
anchor test               # Executer les tests
anchor deploy             # Deploier
anchor run <script>       # Executer un script

CLUSTER:
anchor test --skip-local-validator  # Avec validator externe
anchor deploy --provider.cluster devnet
""")


INSTALLATION:
cargo install --git https://github.com/coral-xyz/anchor avm --locked --force
avm install latest
avm use latest

CREATION PROJET:
anchor init my_program
cd my_program

STRUCTURE:
my_program/
|-- Anchor.toml          # Configuration
|-- Cargo.toml
|-- programs/
|   `-- my_program/
|       |-- Cargo.toml
|       `-- src/
|           `-- lib.rs   # Code principal
|-- tests/
|   `-- my_program.ts    # Tests
`-- app/                 # Frontend

COMMANDES:
anchor build              # Compiler
anchor test               # Executer les tests
anchor deploy             # Deploier
anchor run <script>       # Executer un script

CLUSTER:
anchor test --skip-local-validator  # Avec validator externe
anchor deploy --provider.cluster devnet



---

## 6. Exercices

In [6]:
# Exercice: Token Escrow
EXERCISE_ESCROW = '''
#[program]
pub mod escrow {
    use super::*;

    pub fn make(
        ctx: Context<Make>,
        seed: u64,
        receive_amount: u64,
    ) -> Result<()> {
        let escrow = &mut ctx.accounts.escrow;
        escrow.seed = seed;
        escrow.initializer = ctx.accounts.initializer.key();
        escrow.mint_a = ctx.accounts.mint_a.key();
        escrow.mint_b = ctx.accounts.mint_b.key();
        escrow.receive_amount = receive_amount;

        // Transfer tokens to vault
        token::transfer(
            CpiContext::new(
                ctx.accounts.token_program.to_account_info(),
                token::Transfer {
                    from: ctx.accounts.deposit.to_account_info(),
                    to: ctx.accounts.vault.to_account_info(),
                    authority: ctx.accounts.initializer.to_account_info(),
                },
            ),
            ctx.accounts.deposit.amount,
        )
    }

    pub fn take(ctx: Context<Take>) -> Result<()> {
        // Transfer B to initializer
        token::transfer(
            CpiContext::new(
                ctx.accounts.token_program.to_account_info(),
                token::Transfer {
                    from: ctx.accounts.depositor_token_b.to_account_info(),
                    to: ctx.accounts.initializer_token_b.to_account_info(),
                    authority: ctx.accounts.depositor.to_account_info(),
                },
            ),
            ctx.accounts.escrow.receive_amount,
        )?;

        // Transfer A to depositor
        token::transfer(
            CpiContext::new_with_signer(
                ctx.accounts.token_program.to_account_info(),
                token::Transfer {
                    from: ctx.accounts.vault.to_account_info(),
                    to: ctx.accounts.depositor_token_a.to_account_info(),
                    authority: ctx.accounts.escrow.to_account_info(),
                },
                &[&[
                    b"escrow",
                    ctx.accounts.maker.key().as_ref(),
                    &ctx.accounts.escrow.seed.to_le_bytes()[..],
                    &[ctx.bumps.escrow],
                ]],
            ),
            ctx.accounts.vault.amount,
        )
    }
}
'''
print("Exercice Escrow:")
print(EXERCISE_ESCROW)

Exercice Escrow:

#[program]
pub mod escrow {
    use super::*;

    pub fn make(
        ctx: Context<Make>,
        seed: u64,
        receive_amount: u64,
    ) -> Result<()> {
        let escrow = &mut ctx.accounts.escrow;
        escrow.seed = seed;
        escrow.initializer = ctx.accounts.initializer.key();
        escrow.mint_a = ctx.accounts.mint_a.key();
        escrow.mint_b = ctx.accounts.mint_b.key();
        escrow.receive_amount = receive_amount;

        // Transfer tokens to vault
        token::transfer(
            CpiContext::new(
                ctx.accounts.token_program.to_account_info(),
                token::Transfer {
                    from: ctx.accounts.deposit.to_account_info(),
                    to: ctx.accounts.vault.to_account_info(),
                    authority: ctx.accounts.initializer.to_account_info(),
                },
            ),
            ctx.accounts.deposit.amount,
        )
    }

    pub fn take(ctx: Context<Take>) -> Result<()> {


---

## 7. Resume

| Concept | Description |
|---------|-------------|
| Program | Code executable stateless |
| Account | Stockage de donnees |
| PDA | Adresse derivee du programme |
| CPI | Appel inter-programmes |
| Anchor | Framework Rust pour Solana |

---

**Notebook suivant** : [SC-14-Cross-Chain](SC-14-Cross-Chain.ipynb)